# Q9. Can light fine-tuning of Evo2 improve missense pathogenicity prediction compared with frozen representations and zero-shot scoring?

Missense-only experiment outline; training and evaluation have not been run. [Shared dependencies](../requirements.txt).

## BioNeMo starting point

Use the [pinned NVIDIA tutorial](https://github.com/NVIDIA-BioNeMo/bionemo-recipes/blob/ca16c2acf9bf813d020b6d1e2d4e1240cfef6a69/docs/docs/user-guide/examples/bionemo-evo2/fine-tuning-tutorial.ipynb), reviewed **2026-09-07**, as the training scaffold:

**FASTA → indexed tokens → converted checkpoint → training.**

It demonstrates continued DNA language-model training with all weights trainable. Q9 needs a supervised missense classifier and selective weight updates; a short run alone does not make tuning parameter-efficient.

<details>
<summary>Tutorial settings and implementation references</summary>

- Tutorial example: Evo2 **1B**, one GPU, **1,024** tokens, microbatch **2**, **100** steps, learning rate **1e-4**, **5** warmup steps and validation every **50** steps. These are reference settings, not validated Q9 hyperparameters.
- Entry points: `preprocess_evo2`, `evo2_convert_to_nemo2`, `train_evo2`. The [trainer](https://github.com/NVIDIA-BioNeMo/bionemo-recipes/blob/ca16c2acf9bf813d020b6d1e2d4e1240cfef6a69/sub-packages/bionemo-evo2/src/bionemo/evo2/run/train.py) uses BF16, activation recomputation and Adam with a cosine schedule; it exposes no LoRA or selective-freezing CLI option at this revision.
- Replace the tutorial’s undefined `{preprocessed_data}` interpolation with its defined `output_dir`. Use dedicated `results/q9/` paths and preserve prior runs.
- Source revision: `ca16c2acf9bf813d020b6d1e2d4e1240cfef6a69`. Tutorial SHA-256: `754af9d7ea6c5ea9686445906c30d29c99db72a56cf53ac4c96479c4a5b10b18`.

</details>

## Q9 adaptation

First candidate: train **the final Evo2 block and a binary classification head**, freezing the remaining backbone. Preserve [Q2](Q2-evo2-classifier.ipynb)’s mean pooling, strand averaging and reference/alternate-difference features. Fit a class-weighted binary cross-entropy objective, deriving weights from training labels only.

The paired-variant data loader, classification head, loss and explicit parameter freezing must be added to the BioNeMo scaffold. Log trainable names and counts; verify that only intended weights change. LoRA remains an alternative requiring separate compatibility checks.

## Missense-only inputs and splits

Use the regenerated 5,000-variant missense-only VCFs from [Q1](Q1-clinvar-split.ipynb). Existing split assignments are preserved; Q1 records the new counts, eligibility checks and frozen hashes.

Keep **1,024-base contexts** and seed **42**. Build paired sequence inputs separately for training and validation, retaining variant IDs; do not use the tutorial’s automatic 90/5/5 split or whole-chromosome training data. Reference, alternate and reverse-complement sequences remain together. No independent test partition is introduced.

## Before training

Confirm checkpoint equivalence and complete a one-batch forward/backward check on the **L40S** before choosing a training budget. Check finite loss and gradients, frozen-weight integrity, reload consistency, peak GPU memory and step time. Feasibility is unverified.

<details>
<summary>Checkpoint and software compatibility</summary>

The [converter](https://github.com/NVIDIA-BioNeMo/bionemo-recipes/blob/ca16c2acf9bf813d020b6d1e2d4e1240cfef6a69/sub-packages/bionemo-evo2/src/bionemo/evo2/utils/checkpoint/convert_to_nemo.py) accepts Savanna-format checkpoints; the tutorial uses `arcinstitute/savanna_evo2_1b_base`. Q2 uses `arcinstitute/evo2_1b_base` in Vortex format. Do not substitute filenames or assume equivalence: pin the source weights and tokenizer, audit conversion and compare pre-update predictions under matched precision before the experiment.

The [pinned Dockerfile](https://github.com/NVIDIA-BioNeMo/bionemo-recipes/blob/ca16c2acf9bf813d020b6d1e2d4e1240cfef6a69/Dockerfile) starts from NVIDIA PyTorch **25.01** with Transformer Engine **1.13** and builds BioNeMo/NeMo/Megatron dependencies. Q2 uses **25.04** with Transformer Engine **2.2.0**. BioNeMo, NeMo and Megatron are not installed in the current project environment. Establish and record a compatible training environment without replacing Q2’s stack; this study installs nothing.

</details>

## Leakage checks

Verify missense eligibility, Q1 manifests and VCF hashes before fitting. Recheck shared genes, source IDs, loci, overlapping windows and identical contexts; stop on detected overlap or mismatched inputs. Clinical annotations determine eligibility and labels, not predictor features. Homology, shared patients and pretraining contamination remain unresolved.

## Planned comparison

Fit on training data only; select settings and checkpoints by validation AUROC. Re-establish Q2’s frozen classifier, zero-shot scores and sequence baseline on identical missense inputs. Compare AUROC, average precision and paired component-bootstrap differences; plot training time and peak memory alongside performance. Save artifacts under `results/q9/`.

These are development estimates: validation guides selection, and bootstrap intervals do not remove that bias. Final claims require an untouched holdout.

## Conclusion

BioNeMo provides a concrete training scaffold, but its tutorial does not establish that light tuning improves missense prediction. Selective supervised tuning, checkpoint compatibility and GPU feasibility still need implementation and validation; the research question remains **unanswered**.